In [ ]:
%pip install wandb kaggle optuna -Uq

import torch
if torch.cuda.is_available():
    print(f"GPU found: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU found -- go to Runtime -> Change runtime type -> GPU (T4), then re-run this cell.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

In [ ]:
import os

try:
    from google.colab import userdata
    _kaggle_token = userdata.get("KAGGLE_API_TOKEN")
except Exception:
    _kaggle_token = None

_kaggle_token = _kaggle_token or os.environ.get("KAGGLE_API_TOKEN")
if not _kaggle_token:
    import getpass
    _kaggle_token = getpass.getpass("Paste your Kaggle API token (kaggle.com/settings/api): ")

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
    f.write(_kaggle_token.strip())
os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)

import subprocess
import sys
import zipfile
from pathlib import Path

COMPETITION = "walmart-recruiting-store-sales-forecasting"
_LOCAL_DATA_DIR = Path("data") / COMPETITION
_LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (_LOCAL_DATA_DIR / "train.csv.zip").exists():
    print(f"Downloading '{COMPETITION}' into {_LOCAL_DATA_DIR} ...")
    subprocess.run(
        [sys.executable, "-m", "kaggle", "competitions", "download",
         "-c", COMPETITION, "-p", str(_LOCAL_DATA_DIR)],
        check=True,
    )
    outer_zip = _LOCAL_DATA_DIR / f"{COMPETITION}.zip"
    if outer_zip.exists():
        print(f"Unzipping {outer_zip.name} ...")
        with zipfile.ZipFile(outer_zip) as zf:
            zf.extractall(_LOCAL_DATA_DIR)
        outer_zip.unlink()
else:
    print("Data already present in this session, skipping download.")

for f in sorted(_LOCAL_DATA_DIR.iterdir()):
    print(" -", f.name)

In [ ]:
import getpass
import wandb

try:
    from google.colab import userdata
    _wandb_api_key = userdata.get("WANDB_API_KEY")
except Exception:
    _wandb_api_key = None

_wandb_api_key = _wandb_api_key or os.environ.get("WANDB_API_KEY")

try:
    wandb.logout()
except Exception:
    pass

WANDB_ENTITY = "toberi23-free-university-of-tbilisi-"
WANDB_PROJECT = "Walmart-Recruiting-Store-Sales-Forecasting"
WANDB_GROUP = "PatchTST_Training"

wandb_key = _wandb_api_key or getpass.getpass("Paste your W&B API key (wandb.ai/authorize): ")
wandb.login(key=wandb_key, relogin=True)
print("W&B login OK. Group for this notebook:", WANDB_GROUP)

In [ ]:
import time
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pd.set_option("display.width", 200)

In [ ]:
_LOCAL_DATA_DIR = os.path.join("data", "walmart-recruiting-store-sales-forecasting")
_KAGGLE_DATA_DIR = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"

COMP = (
    os.environ.get("WALMART_DATA_DIR")
    or (_LOCAL_DATA_DIR if os.path.isdir(_LOCAL_DATA_DIR) else _KAGGLE_DATA_DIR)
)
print("Reading competition data from:", COMP)

def load_merged(kind: str = "train") -> pd.DataFrame:
    if kind not in ("train", "test"):
        raise ValueError("kind must be 'train' or 'test'")
    base = pd.read_csv(f"{COMP}/{kind}.csv.zip")
    base["Date"] = pd.to_datetime(base["Date"])
    stores = pd.read_csv(f"{COMP}/stores.csv")
    feats = pd.read_csv(f"{COMP}/features.csv.zip")
    feats["Date"] = pd.to_datetime(feats["Date"])
    feats = feats.drop(columns=["IsHoliday"])
    return (
        base.merge(stores, on="Store", how="left")
            .merge(feats, on=["Store", "Date"], how="left")
            .sort_values(["Store", "Dept", "Date"])
            .reset_index(drop=True)
    )

df_train = load_merged("train")
df_test = load_merged("test")
print("df_train:", df_train.shape, "df_test:", df_test.shape)
print("Unique (Store, Dept) series in train:", df_train.groupby(["Store", "Dept"]).ngroups)

HOLIDAY_BY_DATE = (
    pd.concat([df_train[["Date", "IsHoliday"]], df_test[["Date", "IsHoliday"]]])
      .drop_duplicates("Date")
      .set_index("Date")["IsHoliday"]
)
print(f"HOLIDAY_BY_DATE covers {len(HOLIDAY_BY_DATE)} dates, {int(HOLIDAY_BY_DATE.sum())} of them holiday weeks")

In [ ]:
def wmae(y_true, y_pred, is_holiday):
    w = np.where(np.asarray(is_holiday, dtype=bool), 5.0, 1.0)
    return float(np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w))

assert abs(wmae([10, 20], [8, 15], [True, False]) - 15 / 6) < 1e-9

def wmae_split(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    is_holiday = np.asarray(is_holiday, dtype=bool)
    overall = wmae(y_true, y_pred, is_holiday)
    holiday_wmae = (
        wmae(y_true[is_holiday], y_pred[is_holiday], np.ones(int(is_holiday.sum()), dtype=bool))
        if is_holiday.any() else float("nan")
    )
    nonholiday_wmae = (
        wmae(y_true[~is_holiday], y_pred[~is_holiday], np.zeros(int((~is_holiday).sum()), dtype=bool))
        if (~is_holiday).any() else float("nan")
    )
    return overall, holiday_wmae, nonholiday_wmae

VAL_START = pd.Timestamp("2011-11-04")
VAL_END = pd.Timestamp("2012-07-27")
HORIZON = 39

cv_train = df_train[df_train.Date < VAL_START].copy()
cv_val = df_train[(df_train.Date >= VAL_START) & (df_train.Date <= VAL_END)].copy()

n_val_weeks = cv_val["Date"].nunique()
print("CV train:", cv_train.shape, cv_train.Date.min().date(), "->", cv_train.Date.max().date())
print("CV val  :", cv_val.shape, cv_val.Date.min().date(), "->", cv_val.Date.max().date(), f"({n_val_weeks} weeks)")
assert n_val_weeks == HORIZON, "CV val window must be exactly HORIZON weeks to match the real test set"

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class SeriesBuilder(BaseEstimator, TransformerMixin):

    def __init__(self, min_history_weeks=8):
        self.min_history_weeks = min_history_weeks

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        series = {}
        n_gap_filled = 0
        n_dropped_short = 0
        for (store, dept), g in X.groupby(["Store", "Dept"]):
            g = g.sort_values("Date").drop_duplicates("Date")
            if len(g) < self.min_history_weeks:
                n_dropped_short += 1
                continue
            full_idx = pd.date_range(g["Date"].min(), g["Date"].max(), freq="W-FRI")
            s = g.set_index("Date")["Weekly_Sales"].reindex(full_idx)
            n_gap_filled += int(s.isna().sum())
            s = s.interpolate(method="linear").bfill().ffill()
            series[(store, dept)] = s

        self.n_series_ = len(series)
        self.n_gap_filled_ = n_gap_filled
        self.n_dropped_short_ = n_dropped_short
        return series

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
                  name="PatchTST_Data_Cleaning", job_type="data-cleaning",
                  tags=["patchtst", "data-cleaning"],
                  config={"min_history_weeks": 8})

series_builder = SeriesBuilder(min_history_weeks=8)
series_builder.fit(cv_train)
cv_train_series = series_builder.transform(cv_train)

print(f"Series kept: {series_builder.n_series_}, dropped (too short): {series_builder.n_dropped_short_}")
print(f"Weeks gap-filled by interpolation: {series_builder.n_gap_filled_}")

wandb.log({
    "n_series_kept": series_builder.n_series_,
    "n_series_dropped_short": series_builder.n_dropped_short_,
    "n_weeks_gap_filled": series_builder.n_gap_filled_,
})
run.finish()

In [ ]:
class WindowGenerator:

    def __init__(self, lookback, horizon=HORIZON, stride=2, holiday_by_date=None):
        self.lookback = lookback
        self.horizon = horizon
        self.stride = stride
        self.holiday_by_date = holiday_by_date

    def _holiday_flags(self, dates):
        if self.holiday_by_date is None:
            return np.zeros(len(dates), dtype=bool)
        return self.holiday_by_date.reindex(dates).fillna(False).values.astype(bool)

    def make_training_windows(self, series_dict):
        X, Y, is_holiday, keys = [], [], [], []
        L, H = self.lookback, self.horizon
        for key, s in series_dict.items():
            values = s.values.astype("float32")
            dates = s.index
            last_start = len(values) - L - H
            if last_start < 0:
                continue
            for start in range(0, last_start + 1, self.stride):
                X.append(values[start:start + L])
                Y.append(values[start + L:start + L + H])
                is_holiday.append(self._holiday_flags(dates[start + L:start + L + H]))
                keys.append(key)
        X = np.asarray(X, dtype="float32")
        Y = np.asarray(Y, dtype="float32")
        is_holiday = np.asarray(is_holiday, dtype=bool) if len(is_holiday) else np.zeros((0, H), dtype=bool)
        return X, Y, is_holiday, keys

    def make_forecast_windows(self, series_dict):
        X, keys, last_dates = [], [], []
        L = self.lookback
        for key, s in series_dict.items():
            values = s.values.astype("float32")
            if len(values) >= L:
                window = values[-L:]
            else:
                pad = L - len(values)
                window = np.concatenate([np.zeros(pad, dtype="float32"), values])
            X.append(window)
            keys.append(key)
            last_dates.append(s.index.max())
        X = np.asarray(X, dtype="float32")
        return X, keys, last_dates

def build_val_arrays(series_dict, window_gen, cv_val_df):
    X, keys, _ = window_gen.make_forecast_windows(series_dict)
    val_pivot = cv_val_df.pivot_table(index="Date", columns=["Store", "Dept"], values="Weekly_Sales")
    holiday_by_date = cv_val_df.drop_duplicates("Date").set_index("Date")["IsHoliday"]
    dates = sorted(cv_val_df["Date"].unique())

    Y_true, is_holiday, keep_mask = [], [], []
    for key in keys:
        if key not in val_pivot.columns:
            keep_mask.append(False)
            continue
        row = val_pivot[key].reindex(dates)
        if row.isna().any():
            keep_mask.append(False)
            continue
        Y_true.append(row.values.astype("float32"))
        is_holiday.append(holiday_by_date.reindex(dates).values.astype(bool))
        keep_mask.append(True)

    keep_mask = np.array(keep_mask)
    return X[keep_mask], np.asarray(Y_true), np.asarray(is_holiday)

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
                  name="PatchTST_Feature_Engineering", job_type="feature-engineering",
                  tags=["patchtst", "feature-engineering"],
                  config={"demo_lookback": 52, "demo_stride": 2})

_demo_wg = WindowGenerator(lookback=52, horizon=HORIZON, stride=2, holiday_by_date=HOLIDAY_BY_DATE)
_demo_X, _demo_Y, _demo_is_holiday, _demo_keys = _demo_wg.make_training_windows(cv_train_series)
print(f"Demo (lookback=52, stride=2): {len(_demo_X)} training windows from {len(set(_demo_keys))} series")
print(f"Mean window value: {_demo_X.mean():.1f}, median: {np.median(_demo_X):.1f}")
print(f"Fraction of forecast-horizon weeks flagged as holiday: {_demo_is_holiday.mean():.4f}")

wandb.log({
    "demo_n_windows": len(_demo_X),
    "demo_n_series_with_windows": len(set(_demo_keys)),
    "demo_mean_value": float(_demo_X.mean()),
    "demo_median_value": float(np.median(_demo_X)),
    "demo_holiday_fraction": float(_demo_is_holiday.mean()),
})
run.finish()

In [ ]:
class PatchTST(nn.Module):

    def __init__(self, lookback, horizon, patch_len=16, stride=8, d_model=64, n_heads=4,
                 num_encoder_layers=2, d_ff=256, dropout=0.1, eps=1e-5):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError(f"d_model ({d_model}) must be evenly divisible by n_heads ({n_heads})")
        if patch_len > lookback:
            raise ValueError(f"patch_len ({patch_len}) cannot exceed lookback ({lookback})")
        self.lookback = lookback
        self.horizon = horizon
        self.patch_len = patch_len
        self.stride = stride
        self.eps = eps

        self.pad_len = (-(lookback - patch_len)) % stride
        self.n_patches = (lookback + self.pad_len - patch_len) // stride + 1
        if self.n_patches < 1:
            raise ValueError(f"stride ({stride}) too large for lookback={lookback}, patch_len={patch_len}")

        self.patch_embed = nn.Linear(patch_len, d_model)

        self.pos_embed = nn.Parameter(torch.zeros(1, self.n_patches, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff, dropout=dropout,
            activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)

        self.head_dropout = nn.Dropout(dropout)
        self.head = nn.Linear(self.n_patches * d_model, horizon)

    def forward(self, x):
        mean = x.mean(dim=1, keepdim=True)
        std = x.std(dim=1, keepdim=True) + self.eps
        x_norm = (x - mean) / std

        if self.pad_len > 0:
            x_norm = torch.cat([x_norm, x_norm[:, -1:].expand(-1, self.pad_len)], dim=1)

        patches = x_norm.unfold(dimension=1, size=self.patch_len, step=self.stride)

        tokens = self.patch_embed(patches) + self.pos_embed
        encoded = self.encoder(tokens)

        flat = encoded.reshape(encoded.size(0), -1)
        out_norm = self.head(self.head_dropout(flat))

        return out_norm * std + mean

def build_patchtst(lookback, horizon, patch_len=16, stride=8, d_model=64, n_heads=4,
                    num_encoder_layers=2, d_ff=256, dropout=0.1):
    return PatchTST(lookback, horizon, patch_len=patch_len, stride=stride, d_model=d_model,
                     n_heads=n_heads, num_encoder_layers=num_encoder_layers, d_ff=d_ff, dropout=dropout)

class WindowDataset(Dataset):

    def __init__(self, X, Y, is_holiday):
        self.X = torch.from_numpy(X)
        self.Y = torch.from_numpy(Y)
        self.is_holiday = torch.from_numpy(is_holiday)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], self.is_holiday[idx]

LOSS_FNS = {
    "mse": nn.MSELoss(),
    "mae": nn.L1Loss(),
}

def weighted_mae_loss(pred, target, is_holiday):
    weights = torch.where(is_holiday, torch.tensor(5.0, device=pred.device), torch.tensor(1.0, device=pred.device))
    return (weights * (pred - target).abs()).sum() / weights.sum()

def train_patchtst(model, train_loader, val_X, val_Y_true, val_is_holiday,
                    train_X, train_Y_true, train_is_holiday,
                    learning_rate=1e-3, max_epochs=30, patience=5, loss_fn_name="mae",
                    weight_decay=0.0, verbose=False):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    loss_fn = LOSS_FNS.get(loss_fn_name)

    best_wmae = float("inf")
    best_state = None
    best_epoch = 0
    epochs_without_improvement = 0
    val_X_t = torch.from_numpy(val_X).to(DEVICE)

    history = []
    for epoch in range(max_epochs):
        model.train()
        epoch_loss_sum, epoch_loss_n = 0.0, 0
        for xb, yb, ihb in train_loader:
            xb, yb, ihb = xb.to(DEVICE), yb.to(DEVICE), ihb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            if loss_fn_name == "weighted_mae":
                loss = weighted_mae_loss(pred, yb, ihb)
            else:
                loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            epoch_loss_sum += loss.item() * xb.size(0)
            epoch_loss_n += xb.size(0)
        epoch_train_loss = epoch_loss_sum / max(epoch_loss_n, 1)

        model.eval()
        with torch.no_grad():
            val_pred = model(val_X_t).cpu().numpy()
        epoch_wmae, epoch_holiday_wmae, epoch_nonholiday_wmae = wmae_split(
            val_Y_true.ravel(), val_pred.ravel(), val_is_holiday.ravel()
        )

        epoch_row = {
            "epoch": epoch + 1,
            "train_loss": epoch_train_loss,
            "val_wmae": epoch_wmae,
            "val_holiday_wmae": epoch_holiday_wmae,
            "val_nonholiday_wmae": epoch_nonholiday_wmae,
        }
        history.append(epoch_row)
        if wandb.run is not None:
            wandb.log(epoch_row)

        if verbose:
            print(f"  epoch {epoch + 1}/{max_epochs}: train_loss={epoch_train_loss:.4f}, "
                  f"val WMAE={epoch_wmae:.2f} (holiday={epoch_holiday_wmae:.2f}, non-holiday={epoch_nonholiday_wmae:.2f})")

        if epoch_wmae < best_wmae - 1e-6:
            best_wmae = epoch_wmae
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break

    model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        train_pred = model(torch.from_numpy(train_X).to(DEVICE)).cpu().numpy()
    train_wmae = wmae(train_Y_true.ravel(), train_pred.ravel(), train_is_holiday.ravel())

    return model, best_wmae, best_epoch, train_wmae, history

def train_patchtst_fixed_epochs(model, train_loader, epochs, train_X, train_Y_true, train_is_holiday,
                                 learning_rate=1e-3, loss_fn_name="mae", weight_decay=0.0):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    loss_fn = LOSS_FNS.get(loss_fn_name)

    history = []
    for epoch in range(epochs):
        model.train()
        epoch_loss_sum, epoch_loss_n = 0.0, 0
        for xb, yb, ihb in train_loader:
            xb, yb, ihb = xb.to(DEVICE), yb.to(DEVICE), ihb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            if loss_fn_name == "weighted_mae":
                loss = weighted_mae_loss(pred, yb, ihb)
            else:
                loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            epoch_loss_sum += loss.item() * xb.size(0)
            epoch_loss_n += xb.size(0)
        epoch_train_loss = epoch_loss_sum / max(epoch_loss_n, 1)

        epoch_row = {"epoch": epoch + 1, "train_loss": epoch_train_loss}
        history.append(epoch_row)
        if wandb.run is not None:
            wandb.log(epoch_row)

    model.eval()
    with torch.no_grad():
        train_pred = model(torch.from_numpy(train_X).to(DEVICE)).cpu().numpy()
    train_wmae = wmae(train_Y_true.ravel(), train_pred.ravel(), train_is_holiday.ravel())

    return model, train_wmae, history

In [ ]:
INNER_HORIZON = 7

INNER_HOLIDAY_ANCHORS = [pd.Timestamp("2011-02-11"), pd.Timestamp("2011-09-09")]
N_INNER_FOLDS = 3

WINDOW_STRIDE = 3

_CV_TRAIN_CALENDAR = pd.date_range(cv_train["Date"].min(), cv_train["Date"].max(), freq="W-FRI")

def _fold_origins(lookback, horizon=INNER_HORIZON, n_folds=N_INNER_FOLDS):
    T = len(_CV_TRAIN_CALENDAR)
    positions = []
    for anchor in INNER_HOLIDAY_ANCHORS:
        pos = int(_CV_TRAIN_CALENDAR.searchsorted(anchor))
        if pos >= lookback + horizon and pos + horizon <= T:
            positions.append(pos)
    n_recency = max(n_folds - len(positions), 1)
    positions += [T - horizon * k for k in range(n_recency, 0, -1)]
    positions = sorted(set(p for p in positions if p >= lookback + horizon and p + horizon <= T))
    if not positions:
        raise ValueError(
            f"No feasible walk-forward fold origin for lookback={lookback} -- cv_train "
            f"({T} weeks) is too short to support it."
        )
    return [_CV_TRAIN_CALENDAR[p] for p in positions]

def evaluate_config_rolling(config, series_dict, verbose=False):
    lookback = config["lookback"]
    origins = _fold_origins(lookback)
    train_cutoff = origins[0]

    fold_train_series = {key: s[s.index < train_cutoff] for key, s in series_dict.items()}
    fold_train_series = {
        key: s for key, s in fold_train_series.items() if len(s) >= lookback + INNER_HORIZON
    }

    window_gen = WindowGenerator(lookback=lookback, horizon=INNER_HORIZON, stride=WINDOW_STRIDE,
                                  holiday_by_date=HOLIDAY_BY_DATE)
    X, Y, is_holiday, keys = window_gen.make_training_windows(fold_train_series)
    if len(X) == 0:
        raise ValueError(
            "No walk-forward training windows available -- lookback is too large for the history "
            "available before the earliest inner fold origin."
        )

    loader = DataLoader(WindowDataset(X, Y, is_holiday), batch_size=config["batch_size"], shuffle=True)

    model = build_patchtst(lookback, INNER_HORIZON, patch_len=config["patch_len"], stride=config["stride"],
                            d_model=config["d_model"], n_heads=config["n_heads"],
                            num_encoder_layers=config["num_encoder_layers"], d_ff=config["d_ff"],
                            dropout=config["dropout"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"],
                                  weight_decay=config.get("weight_decay", 0.0))
    loss_fn = LOSS_FNS.get(config["loss_fn"])

    inner_epochs = max(1, min(config["max_epochs"], 15))
    model.train()
    for _ in range(inner_epochs):
        for xb, yb, ihb in loader:
            xb, yb, ihb = xb.to(DEVICE), yb.to(DEVICE), ihb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            if config["loss_fn"] == "weighted_mae":
                loss = weighted_mae_loss(pred, yb, ihb)
            else:
                loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()

    model.eval()
    fold_scores, fold_holiday, fold_nonholiday = [], [], []
    for origin in origins:
        origin_series = {key: s[s.index < origin] for key, s in series_dict.items()}
        origin_series = {key: s for key, s in origin_series.items() if len(s) > 0}
        val_X, val_keys, _ = window_gen.make_forecast_windows(origin_series)
        with torch.no_grad():
            preds = model(torch.from_numpy(val_X).to(DEVICE)).cpu().numpy()
        preds = np.clip(preds, 0, None)

        future_dates = pd.date_range(origin, periods=INNER_HORIZON, freq="W-FRI")
        future_holiday = HOLIDAY_BY_DATE.reindex(future_dates).fillna(False).values.astype(bool)

        y_true, y_pred, is_hol = [], [], []
        for key, pred_row in zip(val_keys, preds):
            true_row = series_dict[key].reindex(future_dates)
            if true_row.isna().any():
                continue
            y_true.append(true_row.values.astype("float32"))
            y_pred.append(pred_row)
            is_hol.append(future_holiday)
        if not y_true:
            continue
        y_true = np.concatenate(y_true)
        y_pred = np.concatenate(y_pred)
        is_hol = np.concatenate(is_hol)
        fold_wmae, fold_hol, fold_nonhol = wmae_split(y_true, y_pred, is_hol)
        fold_scores.append(fold_wmae)
        fold_holiday.append(fold_hol)
        fold_nonholiday.append(fold_nonhol)
        if verbose:
            print(f"    fold origin {origin.date()}: WMAE={fold_wmae:.2f}")

    if not fold_scores:
        raise ValueError("No walk-forward fold produced a scoreable validation set.")

    return {
        "wmae": float(np.mean(fold_scores)),
        "wmae_fold_std": float(np.std(fold_scores)),
        "holiday_wmae": float(np.nanmean(fold_holiday)),
        "nonholiday_wmae": float(np.nanmean(fold_nonholiday)),
        "n_train_windows": len(X),
        "fold_scores": fold_scores,
    }

In [ ]:
_MAX_CV_TRAIN_WEEKS = int((VAL_START - df_train["Date"].min()).days // 7)
_MAX_FEASIBLE_LOOKBACK = _MAX_CV_TRAIN_WEEKS - INNER_HORIZON - 6
print(f"cv_train spans ~{_MAX_CV_TRAIN_WEEKS} weeks -> max feasible lookback for this stage: {_MAX_FEASIBLE_LOOKBACK}")

LOOKBACK_MULTIPLIERS = sorted({
    round(m, 2) for m in [0.33, 0.5, 0.67, 0.85, 1.0, 1.15, 1.33, 1.67, 2.0]
    if m * HORIZON <= _MAX_FEASIBLE_LOOKBACK
})
LOOKBACK_CANDIDATES = [int(round(m * HORIZON)) for m in LOOKBACK_MULTIPLIERS]
assert LOOKBACK_CANDIDATES, (
    "No lookback candidate fits inside cv_train's span -- HORIZON is larger than this project's "
    "fold-B cv_train window can support at all; check VAL_START/df_train.Date.min()."
)
print("Lookback candidates for this run:", LOOKBACK_CANDIDATES)

SAMPLE_SERIES_N = 200
_all_keys = list(cv_train_series.keys())
_rng = np.random.default_rng(SEED)
_sample_keys = set(map(tuple, _rng.choice(np.array(_all_keys, dtype=object), size=SAMPLE_SERIES_N, replace=False)))
cv_train_series_sample = {k: v for k, v in cv_train_series.items() if k in _sample_keys}
print(f"Feature-selection sample: {len(cv_train_series_sample)} series")

FAST_SELECTION_CONFIG = dict(
    patch_len=8, stride=4, d_model=32, n_heads=2, num_encoder_layers=1, d_ff=64, dropout=0.0,
    learning_rate=1e-3, batch_size=256, loss_fn="mae", weight_decay=0.0, max_epochs=15,
)

selection_results = []
for lookback in LOOKBACK_CANDIDATES:
    config = {**FAST_SELECTION_CONFIG, "lookback": lookback}

    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
                      name=f"PatchTST_Feature_Selection_lookback_{lookback}", job_type="feature-selection",
                      tags=["patchtst", "feature-selection"],
                      config={**config, "n_series": len(cv_train_series_sample)})
    t0 = time.time()
    try:
        m = evaluate_config_rolling(config, cv_train_series_sample)
    except ValueError as e:
        print(f"lookback={lookback}: skipped ({e})")
        run.finish()
        continue
    elapsed = time.time() - t0

    wandb.log({
        "wmae": m["wmae"], "wmae_fold_std": m["wmae_fold_std"], "holiday_wmae": m["holiday_wmae"],
        "nonholiday_wmae": m["nonholiday_wmae"], "seconds": elapsed, "n_windows": m["n_train_windows"],
    })
    run.finish()

    selection_results.append({
        "lookback": lookback, "wmae": m["wmae"], "wmae_fold_std": m["wmae_fold_std"],
        "holiday_wmae": m["holiday_wmae"], "nonholiday_wmae": m["nonholiday_wmae"], "seconds": elapsed,
    })
    print(f"lookback={lookback}: WMAE={m['wmae']:.2f}+/-{m['wmae_fold_std']:.2f} "
          f"(holiday={m['holiday_wmae']:.2f}, non-holiday={m['nonholiday_wmae']:.2f}) in {elapsed:.1f}s")

selection_df = pd.DataFrame(selection_results).sort_values("wmae").reset_index(drop=True)
print(selection_df)
LOOKBACK = int(selection_df.loc[0, "lookback"])
print(f"Winning lookback: {LOOKBACK}")

In [ ]:
DEFAULT_PATCHTST_CONFIG = dict(
    lookback=LOOKBACK,
    patch_len=max(4, LOOKBACK // 4), stride=max(2, LOOKBACK // 8),
    d_model=64, n_heads=4, num_encoder_layers=2, d_ff=256, dropout=0.1,
    learning_rate=1e-3, batch_size=256, loss_fn="mae", weight_decay=0.0,
    max_epochs=30, patience=5,
)
print("DEFAULT_PATCHTST_CONFIG:", DEFAULT_PATCHTST_CONFIG)

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
                  name="PatchTST_Training_Baseline", job_type="training",
                  tags=["patchtst", "training"],
                  config={**DEFAULT_PATCHTST_CONFIG, "fold": "walk_forward_inner"})

t0 = time.time()
baseline_rolling = evaluate_config_rolling(DEFAULT_PATCHTST_CONFIG, cv_train_series, verbose=True)
elapsed = time.time() - t0

wandb.log({
    "wmae": baseline_rolling["wmae"], "wmae_fold_std": baseline_rolling["wmae_fold_std"],
    "holiday_wmae": baseline_rolling["holiday_wmae"], "nonholiday_wmae": baseline_rolling["nonholiday_wmae"],
    "seconds": elapsed, "n_windows": baseline_rolling["n_train_windows"],
})
run.finish()
print(f"Baseline walk-forward WMAE ({INNER_HORIZON}-week folds, what the Optuna study below uses to "
      f"score every trial): {baseline_rolling['wmae']:.2f} +/- {baseline_rolling['wmae_fold_std']:.2f} "
      f"(holiday={baseline_rolling['holiday_wmae']:.2f}, non-holiday={baseline_rolling['nonholiday_wmae']:.2f}) "
      f"in {elapsed:.1f}s")
print("Note: the honest, single-fold-B, 39-week WMAE (comparable to Prophet/LightGBM/N-BEATS/DLinear) "
      "is deliberately NOT computed here -- cv_val stays untouched until Chapter 11's one-time final "
      "evaluation, per Chapter 6's markdown.")

In [ ]:
import optuna
from optuna.samplers import TPESampler

N_OPTUNA_TRIALS = 40

def suggest_patchtst_config(trial):
    config = {"lookback": LOOKBACK, "max_epochs": DEFAULT_PATCHTST_CONFIG["max_epochs"]}

    config["patch_len"] = trial.suggest_int("patch_len", 4, min(32, LOOKBACK), step=4)
    config["stride"] = trial.suggest_int("stride", 2, max(2, config["patch_len"] * 2), step=2)
    config["d_model"] = trial.suggest_categorical("d_model", [32, 64, 96, 128, 192, 256])

    valid_heads = [h for h in [1, 2, 4, 8, 16] if config["d_model"] % h == 0]
    config["n_heads"] = trial.suggest_categorical("n_heads", valid_heads)

    config["num_encoder_layers"] = trial.suggest_int("num_encoder_layers", 1, 6)
    config["d_ff"] = trial.suggest_categorical("d_ff", [64, 128, 256, 512, 768])
    config["dropout"] = trial.suggest_float("dropout", 0.0, 0.4, step=0.1)
    config["learning_rate"] = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    config["batch_size"] = trial.suggest_categorical("batch_size", [64, 128, 256, 512, 1024])
    config["loss_fn"] = trial.suggest_categorical("loss_fn", ["mse", "mae"])
    config["weight_decay"] = trial.suggest_categorical(
        "weight_decay", [0.0, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2]
    )
    return config

def optuna_objective(trial):
    config = suggest_patchtst_config(trial)

    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
                      name=f"PatchTST_Optuna_trial_{trial.number:03d}", job_type="optuna-trial",
                      tags=["patchtst", "training", "optuna"],
                      config={**config, "fold": "walk_forward_inner"})
    t0 = time.time()
    try:
        m = evaluate_config_rolling(config, cv_train_series)
    except (ValueError, RuntimeError) as e:
        print(f"trial {trial.number}: pruned ({e})")
        wandb.log({"status": "pruned_invalid_config"})
        run.finish()
        raise optuna.TrialPruned()
    elapsed = time.time() - t0

    wandb.log({
        "wmae": m["wmae"], "wmae_fold_std": m["wmae_fold_std"], "holiday_wmae": m["holiday_wmae"],
        "nonholiday_wmae": m["nonholiday_wmae"], "seconds": elapsed, "n_windows": m["n_train_windows"],
    })
    run.finish()

    print(f"trial {trial.number}: WMAE={m['wmae']:.2f}+/-{m['wmae_fold_std']:.2f} "
          f"(holiday={m['holiday_wmae']:.2f}, non-holiday={m['nonholiday_wmae']:.2f}) in {elapsed:.1f}s "
          f"config={config}")
    return m["wmae"]

study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=SEED),
                             study_name="patchtst_walmart")
study.optimize(optuna_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print(f"\nBest trial: #{study.best_trial.number}, WMAE={study.best_value:.2f}")
print("Best params:", study.best_params)

best_config = {
    "lookback": LOOKBACK,
    "max_epochs": DEFAULT_PATCHTST_CONFIG["max_epochs"],
    "patience": DEFAULT_PATCHTST_CONFIG["patience"],
    **study.best_params,
}
print("best_config from the Optuna study:", best_config)

trials_df = study.trials_dataframe()
run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
                  name="PatchTST_Optuna_Study_Summary", job_type="optuna-summary",
                  tags=["patchtst", "training", "optuna"], config={"n_trials": N_OPTUNA_TRIALS})
wandb.log({
    "best_wmae": study.best_value,
    "n_completed_trials": len(trials_df[trials_df["state"] == "COMPLETE"]),
    "n_pruned_trials": len(trials_df[trials_df["state"] == "PRUNED"]),
    "optuna_trials_table": wandb.Table(dataframe=trials_df),
})
run.finish()

In [ ]:
best_config_weighted = {**best_config, "loss_fn": "weighted_mae"}

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
                  name="PatchTST_Training_WeightedLoss", job_type="training",
                  tags=["patchtst", "training"],
                  config={**best_config_weighted, "fold": "walk_forward_inner"})

weighted_rolling = evaluate_config_rolling(best_config_weighted, cv_train_series, verbose=True)
wandb.log({
    "wmae": weighted_rolling["wmae"], "wmae_fold_std": weighted_rolling["wmae_fold_std"],
    "holiday_wmae": weighted_rolling["holiday_wmae"], "nonholiday_wmae": weighted_rolling["nonholiday_wmae"],
})
run.finish()

mae_rolling = evaluate_config_rolling(best_config, cv_train_series, verbose=True)
print(f"\nbest_config (loss_fn=mae):          wmae={mae_rolling['wmae']:.2f}+/-{mae_rolling['wmae_fold_std']:.2f}, "
      f"holiday_wmae={mae_rolling['holiday_wmae']:.2f}, nonholiday_wmae={mae_rolling['nonholiday_wmae']:.2f}")
print(f"best_config (loss_fn=weighted_mae): wmae={weighted_rolling['wmae']:.2f}+/-{weighted_rolling['wmae_fold_std']:.2f}, "
      f"holiday_wmae={weighted_rolling['holiday_wmae']:.2f}, nonholiday_wmae={weighted_rolling['nonholiday_wmae']:.2f}")

USE_WEIGHTED_LOSS = weighted_rolling["wmae"] < mae_rolling["wmae"]
if USE_WEIGHTED_LOSS:
    best_config = best_config_weighted
    print("\n-> weighted_mae wins on walk-forward WMAE -- adopted as the new best_config for Chapter 11's final fits.")
else:
    print("\n-> weighted_mae did not beat plain mae on walk-forward WMAE -- best_config stays loss_fn='mae'.")

In [ ]:
CHRISTMAS_WEEK52_ANCHORS = {
    2010: pd.Timestamp("2010-12-31"),
    2011: pd.Timestamp("2011-12-30"),
    2012: pd.Timestamp("2012-12-28"),
}
TARGET_YEAR = 2012

class ChristmasWeekShiftAdjuster:

    def __init__(self, bulge_threshold=1.10, christmas_anchors=None, target_year=TARGET_YEAR):
        self.bulge_threshold = bulge_threshold
        self.christmas_anchors = christmas_anchors if christmas_anchors is not None else CHRISTMAS_WEEK52_ANCHORS
        self.target_year = target_year

    def fit(self, train_df):
        self.train_start_ = train_df.groupby(["Store", "Dept"])["Date"].min()
        return self

    def adjust(self, X, preds):
        pred_df = X[["Store", "Dept", "Date"]].copy()
        pred_df["Weekly_Sales"] = preds
        out = preds.copy()

        for year, week52_anchor in self.christmas_anchors.items():
            if year != self.target_year:
                continue
            window_dates = [week52_anchor - pd.Timedelta(weeks=w) for w in range(4, -1, -1)]
            mask_window = pred_df["Date"].isin(window_dates)
            if not mask_window.any():
                continue
            for (store, dept), g in pred_df[mask_window].groupby(["Store", "Dept"]):
                if len(g) < 5:
                    continue
                g = g.sort_values("Date")
                vals = g["Weekly_Sales"].values.copy()
                idx = g.index.values
                mean_other = np.mean(np.concatenate([vals[:3], vals[4:]])) if len(vals) == 5 else np.mean(vals)
                if mean_other <= 0 or vals[3] / mean_other < self.bulge_threshold:
                    continue
                shift_frac = 2.0 / 7.0 if year in (2011,) else 2.5 / 7.0
                moved = vals[3] * shift_frac
                vals[3] -= moved
                vals[4] += moved
                out[idx] = vals
        return out

In [ ]:
class PatchTSTForecastPipeline:
    def __init__(self, lookback, holiday_by_date=None, patch_len=16, stride=8, d_model=64, n_heads=4,
                 num_encoder_layers=2, d_ff=256, dropout=0.1, learning_rate=1e-3, batch_size=256,
                 loss_fn="mae", weight_decay=0.0, window_stride=None, max_epochs=60, patience=8,
                 min_history_weeks=8, horizon=HORIZON, use_christmas_shift=True):
        self.lookback = lookback
        self.holiday_by_date = holiday_by_date
        self.patch_len = patch_len
        self.stride = stride
        self.d_model = d_model
        self.n_heads = n_heads
        self.num_encoder_layers = num_encoder_layers
        self.d_ff = d_ff
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.loss_fn = loss_fn
        self.weight_decay = weight_decay
        self.window_stride = window_stride if window_stride is not None else WINDOW_STRIDE
        self.max_epochs = max_epochs
        self.patience = patience
        self.min_history_weeks = min_history_weeks
        self.horizon = horizon
        self.use_christmas_shift = use_christmas_shift
        self.christmas_adjuster = ChristmasWeekShiftAdjuster() if use_christmas_shift else None

    def _internal_early_stopping_split(self, full_series):
        fit_series, val_keys, val_true = {}, [], []
        for key, s in full_series.items():
            if len(s) < self.lookback + self.horizon + 10:
                fit_series[key] = s
                continue
            fit_series[key] = s.iloc[:-self.horizon]
            val_keys.append(key)
            val_true.append(s.iloc[-self.horizon:].values.astype("float32"))
        val_true = np.asarray(val_true) if val_true else np.zeros((0, self.horizon), dtype="float32")
        return fit_series, val_keys, val_true

    def fit(self, train_df, val_df=None, fixed_epochs=None):
        self.series_builder_ = SeriesBuilder(min_history_weeks=self.min_history_weeks)
        self.series_builder_.fit(train_df)
        self.train_series_ = self.series_builder_.transform(train_df)

        self.sales_floor_ = min((float(s.min()) for s in self.train_series_.values()), default=0.0)

        self.window_gen_ = WindowGenerator(lookback=self.lookback, horizon=self.horizon,
                                            stride=self.window_stride, holiday_by_date=self.holiday_by_date)

        if fixed_epochs is not None:
            X, Y, is_holiday, keys = self.window_gen_.make_training_windows(self.train_series_)
            loader = DataLoader(WindowDataset(X, Y, is_holiday), batch_size=self.batch_size, shuffle=True)
            self.model_ = build_patchtst(self.lookback, self.horizon, patch_len=self.patch_len,
                                          stride=self.stride, d_model=self.d_model, n_heads=self.n_heads,
                                          num_encoder_layers=self.num_encoder_layers, d_ff=self.d_ff,
                                          dropout=self.dropout)
            self.model_, self.train_wmae_, self.history_ = train_patchtst_fixed_epochs(
                self.model_, loader, fixed_epochs, X, Y, is_holiday,
                learning_rate=self.learning_rate, loss_fn_name=self.loss_fn, weight_decay=self.weight_decay,
            )
            self.internal_val_wmae_ = None
            self.best_epoch_ = fixed_epochs

            if self.christmas_adjuster is not None:
                self.christmas_adjuster.fit(train_df)
            return self

        if val_df is not None:
            X, Y, is_holiday, keys = self.window_gen_.make_training_windows(self.train_series_)
            val_X, val_true, val_is_holiday = build_val_arrays(self.train_series_, self.window_gen_, val_df)
        else:
            fit_series, val_keys, val_true = self._internal_early_stopping_split(self.train_series_)
            X, Y, is_holiday, keys = self.window_gen_.make_training_windows(fit_series)
            val_source = {k: fit_series[k] for k in val_keys}
            val_X, _, _ = self.window_gen_.make_forecast_windows(val_source)
            val_is_holiday = np.zeros_like(val_true, dtype=bool)

        loader = DataLoader(WindowDataset(X, Y, is_holiday), batch_size=self.batch_size, shuffle=True)
        self.model_ = build_patchtst(self.lookback, self.horizon, patch_len=self.patch_len,
                                      stride=self.stride, d_model=self.d_model, n_heads=self.n_heads,
                                      num_encoder_layers=self.num_encoder_layers, d_ff=self.d_ff,
                                      dropout=self.dropout)
        self.model_, self.internal_val_wmae_, self.best_epoch_, self.train_wmae_, self.history_ = train_patchtst(
            self.model_, loader, val_X, val_true, val_is_holiday, X, Y, is_holiday,
            learning_rate=self.learning_rate, max_epochs=self.max_epochs, patience=self.patience,
            loss_fn_name=self.loss_fn, weight_decay=self.weight_decay,
        )

        if self.christmas_adjuster is not None:
            self.christmas_adjuster.fit(train_df)
        return self

    def predict(self, test_df):
        X, keys, last_dates = self.window_gen_.make_forecast_windows(self.train_series_)

        self.model_.eval()
        with torch.no_grad():
            preds = self.model_(torch.from_numpy(X).to(DEVICE)).cpu().numpy()
        preds = np.clip(preds, self.sales_floor_, None)

        rows = []
        for (store, dept), last_date, pred_row in zip(keys, last_dates, preds):
            future_dates = pd.date_range(last_date + pd.Timedelta(weeks=1), periods=self.horizon, freq="W-FRI")
            for date, value in zip(future_dates, pred_row):
                rows.append((store, dept, date, value))
        pred_long = pd.DataFrame(rows, columns=["Store", "Dept", "Date", "Weekly_Sales"])

        out = test_df[["Store", "Dept", "Date"]].merge(pred_long, on=["Store", "Dept", "Date"], how="left")
        n_missing = out["Weekly_Sales"].isna().sum()
        if n_missing:
            sd_mean = (
                pd.Series({key: float(s.mean()) for key, s in self.train_series_.items()})
                  .rename_axis(["Store", "Dept"]).rename("_sd_mean").reset_index()
            )
            dept_mean = sd_mean.groupby("Dept")["_sd_mean"].mean().rename("_dept_mean").reset_index()
            global_mean = pred_long["Weekly_Sales"].mean()

            out = out.merge(sd_mean, on=["Store", "Dept"], how="left").merge(dept_mean, on="Dept", how="left")
            fallback = out["_sd_mean"].fillna(out["_dept_mean"]).fillna(global_mean)
            out["Weekly_Sales"] = out["Weekly_Sales"].fillna(fallback)
            out = out.drop(columns=["_sd_mean", "_dept_mean"])

        final_preds = out["Weekly_Sales"].values
        if self.christmas_adjuster is not None:
            final_preds = self.christmas_adjuster.adjust(test_df, final_preds)
        return final_preds, n_missing

In [ ]:
np.random.seed(SEED)
torch.manual_seed(SEED)
final_pipeline = PatchTSTForecastPipeline(holiday_by_date=HOLIDAY_BY_DATE,
                                           use_christmas_shift=False, **best_config)

run_no_shift = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
                           name="PatchTST_Training_Final", job_type="training-full",
                           tags=["patchtst", "training"],
                           config={**best_config, "fold": "B_calendar_aligned", "use_christmas_shift": False})
final_pipeline.fit(cv_train, val_df=cv_val)

cv_val_preds, cv_val_n_missing = final_pipeline.predict(cv_val)
cv_wmae_no_shift, cv_holiday_wmae_no_shift, cv_nonholiday_wmae_no_shift = wmae_split(
    cv_val["Weekly_Sales"].values, cv_val_preds, cv_val["IsHoliday"].values
)
train_val_gap_no_shift = cv_wmae_no_shift - final_pipeline.train_wmae_
train_val_gap_pct_no_shift = 100.0 * train_val_gap_no_shift / final_pipeline.train_wmae_
wandb.log({
    "wmae": cv_wmae_no_shift, "train_wmae": final_pipeline.train_wmae_,
    "holiday_wmae": cv_holiday_wmae_no_shift, "nonholiday_wmae": cv_nonholiday_wmae_no_shift,
    "train_val_wmae_gap": train_val_gap_no_shift, "train_val_wmae_gap_pct": train_val_gap_pct_no_shift,
})
run_no_shift.finish()
print(f"CV WMAE (no shift): {cv_wmae_no_shift:.2f} (train_wmae={final_pipeline.train_wmae_:.2f}, "
      f"gap={train_val_gap_no_shift:+.2f} [{train_val_gap_pct_no_shift:+.1f}%], "
      f"holiday={cv_holiday_wmae_no_shift:.2f}, non-holiday={cv_nonholiday_wmae_no_shift:.2f}) "
      f"({cv_val_n_missing} rows fell back to the global mean)")

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
                  name="PatchTST_Training_Final_ChristmasShift", job_type="training-full",
                  tags=["patchtst", "training"],
                  config={**best_config, "fold": "B_calendar_aligned", "use_christmas_shift": True})

np.random.seed(SEED)
torch.manual_seed(SEED)
final_pipeline_shift = PatchTSTForecastPipeline(holiday_by_date=HOLIDAY_BY_DATE,
                                                 use_christmas_shift=True, **best_config)
final_pipeline_shift.fit(cv_train, val_df=cv_val)
cv_val_preds_shift, _ = final_pipeline_shift.predict(cv_val)
cv_wmae_shift, cv_holiday_wmae_shift, cv_nonholiday_wmae_shift = wmae_split(
    cv_val["Weekly_Sales"].values, cv_val_preds_shift, cv_val["IsHoliday"].values
)

train_val_gap_shift = cv_wmae_shift - final_pipeline_shift.train_wmae_
train_val_gap_pct_shift = 100.0 * train_val_gap_shift / final_pipeline_shift.train_wmae_
wandb.log({
    "wmae": cv_wmae_shift,
    "train_wmae": final_pipeline_shift.train_wmae_,
    "holiday_wmae": cv_holiday_wmae_shift,
    "nonholiday_wmae": cv_nonholiday_wmae_shift,
    "wmae_before_shift": cv_wmae_no_shift,
    "wmae_delta": cv_wmae_shift - cv_wmae_no_shift,
    "train_val_wmae_gap": train_val_gap_shift,
    "train_val_wmae_gap_pct": train_val_gap_pct_shift,
})
run.finish()
print(f"CV WMAE (with shift): {cv_wmae_shift:.2f} (train_wmae={final_pipeline_shift.train_wmae_:.2f}, "
      f"gap={train_val_gap_shift:+.2f} [{train_val_gap_pct_shift:+.1f}%], "
      f"holiday={cv_holiday_wmae_shift:.2f}, non-holiday={cv_nonholiday_wmae_shift:.2f}) "
      f"(expected ~equal to no-shift -- CV's Christmas window is 2011)")

In [ ]:
np.random.seed(SEED)
torch.manual_seed(SEED)
deployed_pipeline = PatchTSTForecastPipeline(holiday_by_date=HOLIDAY_BY_DATE,
                                              use_christmas_shift=True, **best_config)

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
                  name="PatchTST_Pipeline_Save", job_type="save-pipeline",
                  tags=["patchtst", "training"],
                  config={**best_config, "deployed_epochs": final_pipeline_shift.best_epoch_})

deployed_pipeline.fit(df_train, fixed_epochs=final_pipeline_shift.best_epoch_)
print(f"deployed_pipeline trained for {final_pipeline_shift.best_epoch_} epochs "
      f"(the real, cv_val-validated stopping point found by final_pipeline_shift above). "
      f"train_wmae={deployed_pipeline.train_wmae_:.2f}")

_diag_series_builder = SeriesBuilder(min_history_weeks=deployed_pipeline.min_history_weeks)
_diag_series_builder.fit(cv_train)
_diag_cv_train_series = _diag_series_builder.transform(cv_train)
_diag_X, _diag_keys, _diag_last_dates = deployed_pipeline.window_gen_.make_forecast_windows(_diag_cv_train_series)

deployed_pipeline.model_.eval()
with torch.no_grad():
    _diag_preds = deployed_pipeline.model_(torch.from_numpy(_diag_X).to(DEVICE)).cpu().numpy()
_diag_preds = np.clip(_diag_preds, 0, None)

_diag_rows = []
for (store, dept), last_date, pred_row in zip(_diag_keys, _diag_last_dates, _diag_preds):
    future_dates = pd.date_range(last_date + pd.Timedelta(weeks=1), periods=deployed_pipeline.horizon, freq="W-FRI")
    for date, value in zip(future_dates, pred_row):
        _diag_rows.append((store, dept, date, value))
_diag_pred_long = pd.DataFrame(_diag_rows, columns=["Store", "Dept", "Date", "Weekly_Sales"])

_diag_out = cv_val[["Store", "Dept", "Date", "IsHoliday"]].merge(_diag_pred_long, on=["Store", "Dept", "Date"], how="left")
_diag_out["Weekly_Sales"] = _diag_out["Weekly_Sales"].fillna(_diag_pred_long["Weekly_Sales"].mean())

deployed_model_estimated_cv_wmae, deployed_model_estimated_cv_holiday_wmae, deployed_model_estimated_cv_nonholiday_wmae = wmae_split(
    cv_val["Weekly_Sales"].values, _diag_out["Weekly_Sales"].values, cv_val["IsHoliday"].values
)
print(f"Estimated real quality of deployed_pipeline's trained weights (fed cv_train history, scored "
      f"against real cv_val): {deployed_model_estimated_cv_wmae:.2f} "
      f"(holiday={deployed_model_estimated_cv_holiday_wmae:.2f}, non-holiday={deployed_model_estimated_cv_nonholiday_wmae:.2f}) "
      f"(compare to final_pipeline_shift's own cv_wmae_shift={cv_wmae_shift:.2f} -- these describe "
      f"different model objects trained under different conditions, so some gap is expected).")

deployed_pipeline.model_.to("cpu")

with open("patchtst_pipeline.pkl", "wb") as f:
    pickle.dump(deployed_pipeline, f)

deployed_train_val_gap = deployed_model_estimated_cv_wmae - deployed_pipeline.train_wmae_
deployed_train_val_gap_pct = 100.0 * deployed_train_val_gap / deployed_pipeline.train_wmae_
print(f"Train/val WMAE gap: {deployed_train_val_gap:+.2f} [{deployed_train_val_gap_pct:+.1f}%]")

wandb.log({
    "deployed_model_estimated_cv_wmae": deployed_model_estimated_cv_wmae,
    "deployed_model_estimated_cv_holiday_wmae": deployed_model_estimated_cv_holiday_wmae,
    "deployed_model_estimated_cv_nonholiday_wmae": deployed_model_estimated_cv_nonholiday_wmae,
    "train_wmae": deployed_pipeline.train_wmae_,
    "train_val_wmae_gap": deployed_train_val_gap,
    "train_val_wmae_gap_pct": deployed_train_val_gap_pct,
})

artifact = wandb.Artifact("patchtst_forecast_pipeline", type="model",
                           metadata={**best_config, "lookback": LOOKBACK,
                                     "deployed_epochs": final_pipeline_shift.best_epoch_,
                                     "deployed_model_estimated_cv_wmae": deployed_model_estimated_cv_wmae,
                                     "deployed_model_estimated_cv_holiday_wmae": deployed_model_estimated_cv_holiday_wmae,
                                     "deployed_model_estimated_cv_nonholiday_wmae": deployed_model_estimated_cv_nonholiday_wmae,
                                     "train_wmae": deployed_pipeline.train_wmae_,
                                     "train_val_wmae_gap": deployed_train_val_gap,
                                     "train_val_wmae_gap_pct": deployed_train_val_gap_pct})
artifact.add_file("patchtst_pipeline.pkl")
run.log_artifact(artifact)
run.finish()

print("Saved patchtst_pipeline.pkl and logged it as a W&B Artifact.")
print("Download this file from the Colab file browser (or via wandb) before the session ends --")
print("Colab's disk is wiped on disconnect, unlike the local-PyCharm notebooks.")